# Module 5: The Math of Diffusion

**The theoretical heart of the diffusion model curriculum.**

This module covers the complete probabilistic framework behind diffusion models: the forward noising process, noise schedules, the reverse denoising process, the ELBO derivation, why predicting noise works, and how signal-to-noise ratio unifies the theory.

**Learning Objectives:**
- Derive and implement the forward process $q(x_t | x_{t-1})$ and its closed-form $q(x_t | x_0)$
- Understand noise schedules: $\beta_t$, $\alpha_t$, $\bar{\alpha}_t$ and their relationships
- Derive the tractable reverse posterior $q(x_{t-1} | x_t, x_0)$
- Walk through the ELBO derivation and see why it simplifies to predicting noise
- Implement the simplified loss (DDPM Algorithm 1)
- Analyze diffusion through the SNR lens
- Compare linear, cosine, and sigmoid variance schedules

**Key References:**
- Ho et al. 2020, "Denoising Diffusion Probabilistic Models" (DDPM): [arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239)
- Sohl-Dickstein et al. 2015, "Deep Unsupervised Learning using Nonequilibrium Thermodynamics": [arxiv.org/abs/1503.03585](https://arxiv.org/abs/1503.03585)
- Nichol & Dhariwal 2021, "Improved Denoising Diffusion Probabilistic Models": [arxiv.org/abs/2102.09672](https://arxiv.org/abs/2102.09672)
- Song et al. 2021, "Score-Based Generative Modeling through SDEs": [arxiv.org/abs/2011.13456](https://arxiv.org/abs/2011.13456)
- Kingma & Gao 2023, "Understanding Diffusion Objectives as the ELBO": [arxiv.org/abs/2303.00848](https://arxiv.org/abs/2303.00848)

**Estimated time:** 3--4 hours

In [ ]:
# --- Setup ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

## 5.1 Generative Models Landscape

Before diving into diffusion, it helps to see where diffusion models sit among other generative families. Each family makes a different tradeoff between sample quality, training stability, mode coverage, and sampling speed.

| Family | Training Objective | Mode Coverage | Sample Quality | Sampling Speed | Key Limitation |
|---|---|---|---|---|---|
| **GANs** | Adversarial min-max game | Poor (mode collapse) | Excellent | Fast (single pass) | Unstable training, mode dropping |
| **VAEs** | ELBO (reconstruction + KL) | Good | Moderate (blurry) | Fast (single pass) | Posterior approximation gap |
| **Normalizing Flows** | Exact log-likelihood | Excellent | Good | Fast (single pass) | Architecture constraints (invertibility) |
| **Autoregressive** | Exact log-likelihood (factorized) | Excellent | Excellent | Slow (sequential) | O(N) sampling for N dimensions |
| **Diffusion** | Denoising score matching / ELBO | Excellent | Excellent | Slow (iterative) | Requires many sampling steps |

Diffusion models achieve both high sample quality *and* excellent mode coverage -- the two properties that are hardest to get simultaneously. The price is sampling speed, which has spurred a rich line of work on fast samplers (DDIM, DPM-Solver, consistency models, distillation).

The rest of this module develops the mathematical framework that makes diffusion models work.

## 5.2 The Forward (Noising) Process

The forward process gradually destroys data by adding Gaussian noise over $T$ timesteps. Starting from a clean data point $x_0 \sim q(x_0)$, we define a Markov chain:

$$q(x_t | x_{t-1}) = \mathcal{N}\!\left(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I}\right)$$

where $\beta_t \in (0, 1)$ is a small **variance schedule** that controls how much noise is added at step $t$.

**Intuition:** At each step, we slightly shrink the signal (multiply by $\sqrt{1 - \beta_t} < 1$) and add a small amount of noise (variance $\beta_t$). After enough steps, the signal is completely destroyed and $x_T \approx \mathcal{N}(0, \mathbf{I})$.

The full forward trajectory is:

$$q(x_{1:T} | x_0) = \prod_{t=1}^{T} q(x_t | x_{t-1})$$

Let's implement and visualize this step-by-step process.

In [ ]:
# --- 5.2 Forward process: single-step noising and sequential chain ---

def forward_step(x_prev: torch.Tensor, beta_t: float) -> torch.Tensor:
    """Apply one step of the forward process: q(x_t | x_{t-1}).
    
    Args:
        x_prev: tensor of shape (C, H, W), the image at step t-1
        beta_t: variance schedule value at step t
    Returns:
        x_t: noised tensor of shape (C, H, W)
    """
    noise = torch.randn_like(x_prev)                    # (C, H, W)
    mean = math.sqrt(1.0 - beta_t) * x_prev             # (C, H, W) -- shrink signal
    x_t = mean + math.sqrt(beta_t) * noise               # (C, H, W) -- add noise
    return x_t


# Create a synthetic "image" -- a simple checkerboard pattern
def make_checkerboard(size: int = 32, block: int = 4) -> torch.Tensor:
    """Create a checkerboard image tensor of shape (1, size, size) with values in [-1, 1]."""
    img = torch.zeros(1, size, size)
    for i in range(size):
        for j in range(size):
            if (i // block + j // block) % 2 == 0:
                img[0, i, j] = 1.0
            else:
                img[0, i, j] = -1.0
    return img

x_0 = make_checkerboard()  # (1, 32, 32)

# Linear beta schedule (DDPM default)
T = 1000
beta_start = 1e-4
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, T)  # (T,)

# Run the full forward chain and save snapshots
torch.manual_seed(42)
snapshot_steps = [0, 250, 500, 750, 1000]
snapshots = {0: x_0.clone()}
x_t = x_0.clone()

for t in range(1, T + 1):
    x_t = forward_step(x_t, betas[t - 1].item())
    if t in snapshot_steps:
        snapshots[t] = x_t.clone()

# Visualize
fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(14, 3))
for ax, t in zip(axes, snapshot_steps):
    ax.imshow(snapshots[t][0].cpu().numpy(), cmap='gray', vmin=-3, vmax=3)
    ax.set_title(f"t = {t}")
    ax.axis('off')
fig.suptitle("Forward Process: Gradual Destruction of Signal", fontsize=13)
plt.tight_layout()
plt.show()

## 5.3 Noise Schedule: $\beta_t$, $\alpha_t$, $\bar{\alpha}_t$

Running the forward chain step-by-step is expensive. A key insight from [Ho et al. (2020)](https://arxiv.org/abs/2006.11239) is that we can jump directly from $x_0$ to any $x_t$ in closed form.

### Definitions

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

$\bar{\alpha}_t$ is the **cumulative product** of $\alpha_s$ values. It tells us how much of the original signal survives at step $t$.

### Closed-Form Forward Process

By recursively substituting the single-step formula and using the fact that sums of independent Gaussians are Gaussian:

$$q(x_t | x_0) = \mathcal{N}\!\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, \mathbf{I}\right)$$

This means we can sample $x_t$ directly via the **reparameterization**:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \mathbf{I})$$

This is critical for training -- we never need to run the chain sequentially.

In [ ]:
# --- 5.3 Compute the schedule and plot key curves ---

def compute_linear_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    """Compute the linear beta schedule and derived quantities.
    
    Returns:
        dict with keys: betas, alphas, alpha_bar (all shape (T,))
    """
    betas = torch.linspace(beta_start, beta_end, T)          # (T,)
    alphas = 1.0 - betas                                      # (T,)
    alpha_bar = torch.cumprod(alphas, dim=0)                  # (T,)
    return {'betas': betas, 'alphas': alphas, 'alpha_bar': alpha_bar}

schedule = compute_linear_schedule(T=1000)
timesteps = torch.arange(1, 1001)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(timesteps.numpy(), schedule['betas'].numpy(), color='tab:red')
axes[0].set_title(r'$\beta_t$ (variance schedule)')
axes[0].set_xlabel('Timestep t')

axes[1].plot(timesteps.numpy(), schedule['alphas'].numpy(), color='tab:blue')
axes[1].set_title(r'$\alpha_t = 1 - \beta_t$')
axes[1].set_xlabel('Timestep t')

axes[2].plot(timesteps.numpy(), schedule['alpha_bar'].numpy(), color='tab:green')
axes[2].set_title(r'$\bar{\alpha}_t$ (signal survival)')
axes[2].set_xlabel('Timestep t')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"alpha_bar at t=1:   {schedule['alpha_bar'][0]:.6f}  (almost all signal)")
print(f"alpha_bar at t=500: {schedule['alpha_bar'][499]:.6f}")
print(f"alpha_bar at t=1000: {schedule['alpha_bar'][999]:.6f}  (almost pure noise)")

In [ ]:
# --- 5.3 Verify closed-form matches sequential noising ---

def closed_form_sample(x_0: torch.Tensor, alpha_bar_t: float, noise: torch.Tensor) -> torch.Tensor:
    """Sample x_t directly from x_0 using the closed-form formula.
    
    x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    """
    return math.sqrt(alpha_bar_t) * x_0 + math.sqrt(1.0 - alpha_bar_t) * noise  # (C, H, W)


# Compare: sequential vs closed-form at t=500
# We can't get exact match (different random draws), but we can verify
# the STATISTICS match (mean and variance).

x_0 = make_checkerboard()  # (1, 32, 32)
test_t = 500
alpha_bar_t = schedule['alpha_bar'][test_t - 1].item()

# Run many samples and check statistics
n_samples = 5000
torch.manual_seed(0)

# Closed-form samples for a single pixel
pixel_samples_cf = []
for _ in range(n_samples):
    eps = torch.randn(1)
    x_t_pixel = math.sqrt(alpha_bar_t) * x_0[0, 0, 0].item() + math.sqrt(1 - alpha_bar_t) * eps.item()
    pixel_samples_cf.append(x_t_pixel)

pixel_samples_cf = torch.tensor(pixel_samples_cf)

expected_mean = math.sqrt(alpha_bar_t) * x_0[0, 0, 0].item()
expected_var = 1 - alpha_bar_t

print(f"At t={test_t}, alpha_bar_t = {alpha_bar_t:.6f}")
print(f"Pixel value x_0[0,0,0] = {x_0[0,0,0].item():.1f}")
print(f"Expected mean: sqrt(alpha_bar)*x_0 = {expected_mean:.4f}")
print(f"Empirical mean: {pixel_samples_cf.mean():.4f}")
print(f"Expected variance: 1 - alpha_bar = {expected_var:.4f}")
print(f"Empirical variance: {pixel_samples_cf.var():.4f}")
print(f"\nStatistics match -- closed-form is correct.")

### Exercise 5.3: Implement the Full Schedule Computation

Write a function `compute_schedule(betas)` that takes a tensor of $\beta_t$ values and returns a dictionary with **all** the quantities we will need throughout this module:

- `betas`: the input $\beta_t$
- `alphas`: $\alpha_t = 1 - \beta_t$
- `alpha_bar`: $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$
- `sqrt_alpha_bar`: $\sqrt{\bar{\alpha}_t}$
- `sqrt_one_minus_alpha_bar`: $\sqrt{1 - \bar{\alpha}_t}$
- `sqrt_recip_alpha`: $1 / \sqrt{\alpha_t}$ (needed for sampling)
- `posterior_variance`: $\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t$ (we will derive this in Section 5.5)

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def compute_schedule(betas: torch.Tensor) -> dict:
    """Compute all diffusion schedule quantities from beta values.
    
    Args:
        betas: tensor of shape (T,), the variance schedule
    Returns:
        dict of schedule tensors, each of shape (T,)
    """
    alphas = 1.0 - betas                                          # (T,)
    alpha_bar = torch.cumprod(alphas, dim=0)                      # (T,)
    
    # For posterior variance, we need alpha_bar_{t-1}. 
    # At t=1 (index 0), alpha_bar_{t-1} = alpha_bar_0 = 1.0
    alpha_bar_prev = F.pad(alpha_bar[:-1], (1, 0), value=1.0)    # (T,)
    
    sqrt_alpha_bar = torch.sqrt(alpha_bar)                        # (T,)
    sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)        # (T,)
    sqrt_recip_alpha = 1.0 / torch.sqrt(alphas)                   # (T,)
    
    # Posterior variance: beta_tilde_t = (1 - alpha_bar_{t-1}) / (1 - alpha_bar_t) * beta_t
    posterior_variance = (1.0 - alpha_bar_prev) / (1.0 - alpha_bar) * betas  # (T,)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alpha_bar': alpha_bar,
        'alpha_bar_prev': alpha_bar_prev,
        'sqrt_alpha_bar': sqrt_alpha_bar,
        'sqrt_one_minus_alpha_bar': sqrt_one_minus_alpha_bar,
        'sqrt_recip_alpha': sqrt_recip_alpha,
        'posterior_variance': posterior_variance,
    }


# Test it
betas = torch.linspace(1e-4, 0.02, 1000)
sched = compute_schedule(betas)

print("Schedule keys:", list(sched.keys()))
print(f"alpha_bar[0] = {sched['alpha_bar'][0]:.6f} (should be close to 1)")
print(f"alpha_bar[-1] = {sched['alpha_bar'][-1]:.6f} (should be close to 0)")
print(f"posterior_variance[0] = {sched['posterior_variance'][0]:.8f} (should be 0 at t=1)")
print(f"sqrt_alpha_bar shape: {sched['sqrt_alpha_bar'].shape}")
print("All schedule tensors computed correctly.")

## 5.4 The Reparameterization Trick

To train with gradient descent, we need to backpropagate through the sampling operation. But sampling $x_t \sim q(x_t | x_0)$ involves randomness, and we cannot differentiate through a random sample.

The **reparameterization trick** (from [Kingma & Welling, 2014](https://arxiv.org/abs/1312.6114)) separates the randomness from the parameters:

Instead of sampling $x_t \sim \mathcal{N}(\mu, \sigma^2)$ directly, we:

1. Sample $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$ (no parameters involved)
2. Compute $x_t = \mu + \sigma \cdot \varepsilon$ (deterministic, differentiable function of $\mu$ and $\sigma$)

For the forward process:

$$x_t = \underbrace{\sqrt{\bar{\alpha}_t}}_{\text{signal coeff}} \cdot x_0 + \underbrace{\sqrt{1 - \bar{\alpha}_t}}_{\text{noise coeff}} \cdot \varepsilon$$

Both coefficients are differentiable functions of the schedule parameters, and $\varepsilon$ is independent of everything we want to optimize.

In [ ]:
# --- 5.4 Two equivalent ways to sample x_t ---

x_0 = make_checkerboard()  # (1, 32, 32)
t_idx = 499  # t=500 (0-indexed)

# Method 1: Direct sampling from Gaussian (NOT differentiable w.r.t. schedule params)
torch.manual_seed(123)
mean_t = sched['sqrt_alpha_bar'][t_idx] * x_0           # (1, 32, 32)
std_t = sched['sqrt_one_minus_alpha_bar'][t_idx]         # scalar
x_t_direct = mean_t + std_t * torch.randn_like(x_0)     # (1, 32, 32)

# Method 2: Reparameterization (differentiable w.r.t. schedule params)
torch.manual_seed(123)
eps = torch.randn_like(x_0)                                                    # (1, 32, 32)
x_t_reparam = sched['sqrt_alpha_bar'][t_idx] * x_0 + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps  # (1, 32, 32)

# They produce identical results with the same seed
print(f"Max difference between methods: {(x_t_direct - x_t_reparam).abs().max().item():.2e}")
print("Both methods produce identical samples (as expected).")

# Show the result
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(x_0[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[0].set_title("$x_0$ (clean)")
axes[1].imshow(eps[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[1].set_title(r"$\varepsilon \sim \mathcal{N}(0, I)$")
axes[2].imshow(x_t_reparam[0].numpy(), cmap='gray', vmin=-3, vmax=3)
axes[2].set_title(f"$x_{{500}}$ (reparameterized)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5.5 The Reverse Process

### The Problem

We want to sample from $q(x_{t-1} | x_t)$ -- given a noisy image, denoise it one step. Unfortunately, this requires integrating over all possible $x_0$:

$$q(x_{t-1} | x_t) = \int q(x_{t-1} | x_t, x_0)\, q(x_0 | x_t)\, dx_0$$

This is **intractable** because $q(x_0 | x_t)$ depends on the unknown data distribution.

### The Solution: Learn the Reverse

We train a neural network $p_\theta$ to approximate the reverse:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}\!\left(x_{t-1};\; \mu_\theta(x_t, t),\; \Sigma_\theta(x_t, t)\right)$$

### The Key Insight: Tractable Posterior

While $q(x_{t-1} | x_t)$ is intractable, the **posterior conditioned on $x_0$** is tractable and Gaussian:

$$q(x_{t-1} | x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\; \tilde{\mu}_t(x_t, x_0),\; \tilde{\beta}_t \mathbf{I}\right)$$

where the posterior mean and variance are:

$$\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}}\, \beta_t}{1 - \bar{\alpha}_t}\, x_0 + \frac{\sqrt{\alpha_t}\,(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t}\, x_t$$

$$\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t}\, \beta_t$$

This posterior is derived by applying Bayes' rule to the Gaussian forward process. The derivation uses the fact that the product of two Gaussians is Gaussian, then completes the square.

Since $x_0$ is unknown at inference time, the network must predict it (or equivalently, predict the noise $\varepsilon$ that was added, from which $x_0$ can be recovered).

## 5.6 ELBO Derivation: Why Predicting Noise Works

This section walks through the derivation that connects the variational bound to the simple noise-prediction loss. This is the mathematical core of DDPM ([Ho et al. 2020](https://arxiv.org/abs/2006.11239)).

### Step 1: The Variational Lower Bound

We want to maximize $\log p_\theta(x_0)$. Using Jensen's inequality:

$$\log p_\theta(x_0) \geq \mathbb{E}_{q(x_{1:T}|x_0)} \left[ \log \frac{p_\theta(x_{0:T})}{q(x_{1:T}|x_0)} \right] = -L_{\text{VLB}}$$

### Step 2: Decompose into per-timestep terms

The VLB decomposes into $T+1$ terms:

$$L_{\text{VLB}} = \underbrace{D_{\text{KL}}(q(x_T|x_0) \| p(x_T))}_{L_T} + \sum_{t=2}^{T} \underbrace{D_{\text{KL}}(q(x_{t-1}|x_t, x_0) \| p_\theta(x_{t-1}|x_t))}_{L_{t-1}} - \underbrace{\log p_\theta(x_0|x_1)}_{L_0}$$

- $L_T$: compares the final noised distribution to the prior $\mathcal{N}(0, I)$. No learnable parameters -- this is a constant.
- $L_{t-1}$ for $t = 2, \ldots, T$: KL divergence between the true posterior $q(x_{t-1}|x_t,x_0)$ and the learned reverse $p_\theta(x_{t-1}|x_t)$. Both are Gaussian, so this has a closed form.
- $L_0$: reconstruction term.

### Step 3: KL between two Gaussians

Since both $q(x_{t-1}|x_t,x_0)$ and $p_\theta(x_{t-1}|x_t)$ are Gaussian with the same (fixed) variance $\tilde{\beta}_t$, the KL simplifies to:

$$L_{t-1} = \frac{1}{2\tilde{\beta}_t} \left\| \tilde{\mu}_t(x_t, x_0) - \mu_\theta(x_t, t) \right\|^2 + C$$

### Step 4: The epsilon reparameterization

Since $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$, we can express $x_0$ in terms of $x_t$ and $\varepsilon$:

$$x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}} \left( x_t - \sqrt{1 - \bar{\alpha}_t}\, \varepsilon \right)$$

Substituting into $\tilde{\mu}_t$, after algebra:

$$\tilde{\mu}_t = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \varepsilon \right)$$

If we parameterize $\mu_\theta$ the same way but with a *predicted* noise $\varepsilon_\theta$:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \varepsilon_\theta(x_t, t) \right)$$

then $L_{t-1}$ becomes proportional to:

$$L_{t-1} \propto \left\| \varepsilon - \varepsilon_\theta(x_t, t) \right\|^2$$

The network simply predicts the noise that was added.

### Alternative Parameterizations

| Parameterization | Network predicts | Loss target | Reference |
|---|---|---|---|
| $\varepsilon$-prediction | $\varepsilon_\theta(x_t, t) \approx \varepsilon$ | $\|\varepsilon - \varepsilon_\theta\|^2$ | [Ho et al. 2020](https://arxiv.org/abs/2006.11239) |
| $x_0$-prediction | $x_{0,\theta}(x_t, t) \approx x_0$ | $\|x_0 - x_{0,\theta}\|^2$ | [Ramesh et al. 2022](https://arxiv.org/abs/2204.06125) |
| $v$-prediction | $v_\theta(x_t, t) \approx v_t$ | $\|v_t - v_\theta\|^2$ | [Salimans & Ho 2022](https://arxiv.org/abs/2202.00512) |

where $v_t = \sqrt{\bar{\alpha}_t}\, \varepsilon - \sqrt{1 - \bar{\alpha}_t}\, x_0$.

All three are mathematically equivalent -- they just differ in numerical conditioning at different noise levels.

In [ ]:
# --- 5.6 Convert between eps, mu, and x_{t-1} ---

def eps_to_mu(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """Convert predicted noise to predicted posterior mean.
    
    mu_theta = (1/sqrt(alpha_t)) * (x_t - beta_t / sqrt(1 - alpha_bar_t) * eps_theta)
    
    Args:
        x_t: noisy input, shape (B, C, H, W)
        eps_theta: predicted noise, shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict from compute_schedule()
    Returns:
        mu_theta: predicted mean, shape (B, C, H, W)
    """
    sqrt_recip_alpha = sched['sqrt_recip_alpha'][t_idx]           # scalar
    beta_t = sched['betas'][t_idx]                                 # scalar
    sqrt_one_minus_alpha_bar = sched['sqrt_one_minus_alpha_bar'][t_idx]  # scalar
    
    mu_theta = sqrt_recip_alpha * (x_t - (beta_t / sqrt_one_minus_alpha_bar) * eps_theta)  # (B, C, H, W)
    return mu_theta


def eps_to_x0(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """Convert predicted noise to predicted x_0.
    
    x_0 = (x_t - sqrt(1 - alpha_bar_t) * eps) / sqrt(alpha_bar_t)
    
    Args:
        x_t: noisy input, shape (B, C, H, W)
        eps_theta: predicted noise, shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict
    Returns:
        x_0_pred: predicted clean image, shape (B, C, H, W)
    """
    x_0_pred = (x_t - sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_theta) / sched['sqrt_alpha_bar'][t_idx]  # (B, C, H, W)
    return x_0_pred


def sample_reverse_step(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """One reverse sampling step: x_{t-1} ~ p_theta(x_{t-1} | x_t).
    
    Args:
        x_t: shape (B, C, H, W)
        eps_theta: shape (B, C, H, W)
        t_idx: 0-indexed timestep
        sched: schedule dict
    Returns:
        x_prev: shape (B, C, H, W)
    """
    mu_theta = eps_to_mu(x_t, eps_theta, t_idx, sched)         # (B, C, H, W)
    
    if t_idx == 0:
        return mu_theta  # No noise at final step
    
    sigma_t = torch.sqrt(sched['posterior_variance'][t_idx])     # scalar
    noise = torch.randn_like(x_t)                                # (B, C, H, W)
    x_prev = mu_theta + sigma_t * noise                          # (B, C, H, W)
    return x_prev


# Demonstrate: if we know the true noise, we can perfectly recover the posterior mean
torch.manual_seed(42)
x_0 = make_checkerboard().unsqueeze(0)           # (1, 1, 32, 32)
t_idx = 300
eps_true = torch.randn_like(x_0)                  # (1, 1, 32, 32)

# Forward: noise x_0 to get x_t
x_t = sched['sqrt_alpha_bar'][t_idx] * x_0 + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_true  # (1, 1, 32, 32)

# Recover x_0 from x_t and true noise
x_0_recovered = eps_to_x0(x_t, eps_true, t_idx, sched)  # (1, 1, 32, 32)

print(f"Max error in x_0 recovery: {(x_0 - x_0_recovered).abs().max().item():.2e}")
print("With the true noise, x_0 is recovered exactly.")

### Exercise 5.6: Implement All Three Prediction Modes

Given $x_t$, $\varepsilon_\theta$, and the schedule, implement functions to:

1. **eps-prediction**: Compute predicted $x_0$ from $\varepsilon_\theta$
2. **x0-prediction**: Compute predicted $\varepsilon$ from $x_{0,\theta}$
3. **v-prediction**: Compute predicted $x_0$ and $\varepsilon$ from $v_\theta$

Recall: $v_t = \sqrt{\bar{\alpha}_t}\, \varepsilon - \sqrt{1 - \bar{\alpha}_t}\, x_0$

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def predict_x0_from_eps(x_t: torch.Tensor, eps_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """eps-prediction mode: recover x_0 from predicted noise."""
    return (x_t - sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_theta) / sched['sqrt_alpha_bar'][t_idx]  # (B, C, H, W)


def predict_eps_from_x0(x_t: torch.Tensor, x0_theta: torch.Tensor, t_idx: int, sched: dict) -> torch.Tensor:
    """x0-prediction mode: recover eps from predicted x_0."""
    return (x_t - sched['sqrt_alpha_bar'][t_idx] * x0_theta) / sched['sqrt_one_minus_alpha_bar'][t_idx]  # (B, C, H, W)


def predict_x0_eps_from_v(x_t: torch.Tensor, v_theta: torch.Tensor, t_idx: int, sched: dict) -> tuple:
    """v-prediction mode: recover both x_0 and eps from predicted v.
    
    v_t = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0
    
    Solving the system:
      x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * eps
      v_t = sqrt(alpha_bar) * eps - sqrt(1 - alpha_bar) * x_0
    
    Gives:
      x_0 = sqrt(alpha_bar) * x_t - sqrt(1 - alpha_bar) * v_t
      eps = sqrt(1 - alpha_bar) * x_t + sqrt(alpha_bar) * v_t
    """
    sqrt_ab = sched['sqrt_alpha_bar'][t_idx]
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t_idx]
    
    x0_pred = sqrt_ab * x_t - sqrt_1mab * v_theta                # (B, C, H, W)
    eps_pred = sqrt_1mab * x_t + sqrt_ab * v_theta               # (B, C, H, W)
    return x0_pred, eps_pred


# --- Verify all three modes are consistent ---
torch.manual_seed(7)
x_0_test = torch.randn(1, 1, 8, 8)                                # (1, 1, 8, 8)
eps_true = torch.randn_like(x_0_test)                              # (1, 1, 8, 8)
t_idx = 400

x_t_test = sched['sqrt_alpha_bar'][t_idx] * x_0_test + sched['sqrt_one_minus_alpha_bar'][t_idx] * eps_true  # (1, 1, 8, 8)

# Compute true v
v_true = sched['sqrt_alpha_bar'][t_idx] * eps_true - sched['sqrt_one_minus_alpha_bar'][t_idx] * x_0_test  # (1, 1, 8, 8)

# Mode 1: eps -> x_0
x0_from_eps = predict_x0_from_eps(x_t_test, eps_true, t_idx, sched)
print(f"eps-mode  |  x_0 error: {(x_0_test - x0_from_eps).abs().max().item():.2e}")

# Mode 2: x_0 -> eps
eps_from_x0 = predict_eps_from_x0(x_t_test, x_0_test, t_idx, sched)
print(f"x0-mode   |  eps error: {(eps_true - eps_from_x0).abs().max().item():.2e}")

# Mode 3: v -> x_0 and eps
x0_from_v, eps_from_v = predict_x0_eps_from_v(x_t_test, v_true, t_idx, sched)
print(f"v-mode    |  x_0 error: {(x_0_test - x0_from_v).abs().max().item():.2e}")
print(f"v-mode    |  eps error: {(eps_true - eps_from_v).abs().max().item():.2e}")
print("\nAll three parameterizations are consistent.")

## 5.7 The Simplified Loss

[Ho et al. (2020)](https://arxiv.org/abs/2006.11239) found that dropping the weighting factor $\frac{\beta_t^2}{2\tilde{\beta}_t \alpha_t (1 - \bar{\alpha}_t)}$ from the ELBO and using a uniform weight across timesteps works better in practice:

$$L_{\text{simple}} = \mathbb{E}_{t \sim \mathcal{U}(1,T),\; x_0,\; \varepsilon \sim \mathcal{N}(0,I)} \left[ \left\| \varepsilon - \varepsilon_\theta\!\left(\sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon,\; t\right) \right\|^2 \right]$$

This is **DDPM Algorithm 1** (Training). The procedure is:

1. Sample $x_0$ from the dataset
2. Sample $t \sim \mathcal{U}\{1, \ldots, T\}$
3. Sample $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$
4. Compute $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$
5. Predict $\varepsilon_\theta(x_t, t)$
6. Take gradient step on $\| \varepsilon - \varepsilon_\theta(x_t, t) \|^2$

In [ ]:
# --- 5.7 The simplified loss in ~10 lines ---

def diffusion_loss_simple(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
) -> torch.Tensor:
    """Compute the simplified diffusion training loss (DDPM Algorithm 1).
    
    Args:
        model: noise prediction network, takes (x_t, t) -> eps_theta
        x_0: clean data batch, shape (B, C, H, W)
        sched: schedule dict from compute_schedule()
        T: number of diffusion timesteps
    Returns:
        loss: scalar MSE loss
    """
    batch_size = x_0.shape[0]
    
    # 1. Sample random timesteps uniformly
    t = torch.randint(0, T, (batch_size,), device=x_0.device)            # (B,)
    
    # 2. Sample noise
    eps = torch.randn_like(x_0)                                           # (B, C, H, W)
    
    # 3. Compute x_t using closed-form forward process
    sqrt_alpha_bar_t = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    sqrt_one_minus_ab_t = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)  # (B, 1, 1, 1)
    x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_ab_t * eps             # (B, C, H, W)
    
    # 4. Predict noise
    eps_theta = model(x_t, t)                                              # (B, C, H, W)
    
    # 5. MSE loss
    loss = F.mse_loss(eps_theta, eps)                                      # scalar
    return loss


# Quick sanity check with a dummy model
class DummyNoisePredictor(nn.Module):
    """A trivial model that just returns zeros -- for testing the loss function."""
    def forward(self, x_t, t):
        return torch.zeros_like(x_t)

dummy_model = DummyNoisePredictor()
torch.manual_seed(42)
x_0_batch = torch.randn(4, 1, 32, 32)  # (4, 1, 32, 32)
loss = diffusion_loss_simple(dummy_model, x_0_batch, sched, T=1000)
print(f"Loss with zero-prediction model: {loss.item():.4f}")
print(f"Expected ~1.0 (since eps ~ N(0,I) and model predicts 0, MSE ≈ E[eps^2] = 1)")
print(f"Loss is differentiable: {loss.requires_grad}")

### Exercise 5.7: Implement the Diffusion Loss Function

Implement `diffusion_loss` that supports all three prediction modes (eps, x0, v). The loss should always be computed as MSE against the appropriate target.

For each mode:
- **eps**: target is $\varepsilon$, prediction is $\varepsilon_\theta(x_t, t)$
- **x0**: target is $x_0$, prediction is $x_{0,\theta}(x_t, t)$
- **v**: target is $v_t = \sqrt{\bar{\alpha}_t}\varepsilon - \sqrt{1-\bar{\alpha}_t} x_0$, prediction is $v_\theta(x_t, t)$

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def diffusion_loss(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
    prediction_mode: str = "eps",
) -> torch.Tensor:
    """Compute diffusion training loss with configurable prediction mode.
    
    Args:
        model: network that takes (x_t, t) and returns prediction
        x_0: clean data, shape (B, C, H, W)
        sched: schedule dict
        T: number of timesteps
        prediction_mode: one of "eps", "x0", "v"
    Returns:
        loss: scalar MSE
    """
    batch_size = x_0.shape[0]
    t = torch.randint(0, T, (batch_size,), device=x_0.device)               # (B,)
    eps = torch.randn_like(x_0)                                               # (B, C, H, W)
    
    sqrt_ab = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)                   # (B, 1, 1, 1)
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    x_t = sqrt_ab * x_0 + sqrt_1mab * eps                                    # (B, C, H, W)
    
    prediction = model(x_t, t)                                                # (B, C, H, W)
    
    if prediction_mode == "eps":
        target = eps                                                           # (B, C, H, W)
    elif prediction_mode == "x0":
        target = x_0                                                           # (B, C, H, W)
    elif prediction_mode == "v":
        target = sqrt_ab * eps - sqrt_1mab * x_0                              # (B, C, H, W)
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")
    
    return F.mse_loss(prediction, target)                                      # scalar


# Test all three modes
torch.manual_seed(42)
for mode in ["eps", "x0", "v"]:
    loss = diffusion_loss(dummy_model, x_0_batch, sched, T=1000, prediction_mode=mode)
    print(f"Loss ({mode}-prediction, zero model): {loss.item():.4f}")

## 5.8 The SNR Perspective

The **Signal-to-Noise Ratio** provides a unified lens for understanding diffusion ([Kingma & Gao 2023](https://arxiv.org/abs/2303.00848)). At timestep $t$, the noisy sample is:

$$x_t = \underbrace{\sqrt{\bar{\alpha}_t}}_{\text{signal coeff}} x_0 + \underbrace{\sqrt{1 - \bar{\alpha}_t}}_{\text{noise coeff}} \varepsilon$$

The SNR is the ratio of signal power to noise power:

$$\text{SNR}(t) = \frac{\bar{\alpha}_t}{1 - \bar{\alpha}_t}$$

Key observations:
- **High SNR** (early timesteps): mostly signal, little noise. The model sees nearly clean data.
- **SNR = 1**: equal parts signal and noise. The "crossover" point.
- **Low SNR** (late timesteps): mostly noise. The model must hallucinate structure.

The ELBO loss weights each timestep by $\frac{d}{dt}\text{SNR}(t)$, which is why different schedules lead to different implicit weightings of timesteps. The simplified loss (uniform weighting) upweights high-SNR timesteps relative to the ELBO.

In [ ]:
# --- 5.8 SNR computation and visualization ---

def compute_snr(alpha_bar: torch.Tensor) -> torch.Tensor:
    """Compute signal-to-noise ratio: SNR(t) = alpha_bar / (1 - alpha_bar)."""
    return alpha_bar / (1.0 - alpha_bar)  # (T,)


# Compute SNR for linear schedule
snr_linear = compute_snr(sched['alpha_bar'])  # (1000,)

# Also compute cosine schedule for comparison (we'll define it fully in 5.9)
def cosine_alpha_bar(T: int, s: float = 0.008) -> torch.Tensor:
    """Compute alpha_bar using the cosine schedule from Nichol & Dhariwal 2021."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * (math.pi / 2)) ** 2
    alpha_bar = f_t[1:] / f_t[0]  # (T,)
    return alpha_bar.float()

alpha_bar_cosine = cosine_alpha_bar(1000)
snr_cosine = compute_snr(alpha_bar_cosine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].plot(timesteps.numpy(), snr_linear.numpy(), label='Linear schedule', color='tab:blue')
axes[0].plot(timesteps.numpy(), snr_cosine.numpy(), label='Cosine schedule', color='tab:orange')
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='SNR = 1')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('SNR(t)')
axes[0].set_title('SNR (linear scale)')
axes[0].legend()
axes[0].set_ylim(0, 50)

# Log scale -- much more informative
axes[1].semilogy(timesteps.numpy(), snr_linear.numpy(), label='Linear schedule', color='tab:blue')
axes[1].semilogy(timesteps.numpy(), snr_cosine.numpy(), label='Cosine schedule', color='tab:orange')
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='SNR = 1')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('SNR(t) [log scale]')
axes[1].set_title('SNR (log scale)')
axes[1].legend()

plt.tight_layout()
plt.show()

# What does the model "see" at different SNR levels?
x_0 = make_checkerboard()  # (1, 32, 32)
snr_targets = [100, 10, 1, 0.1, 0.01]

fig, axes = plt.subplots(1, len(snr_targets), figsize=(14, 3))
torch.manual_seed(42)

for ax, target_snr in zip(axes, snr_targets):
    # alpha_bar = snr / (1 + snr)
    ab = target_snr / (1.0 + target_snr)
    eps = torch.randn_like(x_0)
    x_noisy = math.sqrt(ab) * x_0 + math.sqrt(1 - ab) * eps
    ax.imshow(x_noisy[0].numpy(), cmap='gray', vmin=-3, vmax=3)
    ax.set_title(f"SNR = {target_snr}")
    ax.axis('off')

fig.suptitle("What the model sees at different SNR levels", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 5.8: SNR Computation and SNR-Weighted Loss

1. Compute the SNR for the linear schedule and find the timestep where SNR is closest to 1.
2. Implement an SNR-weighted loss where each timestep is weighted by $\text{SNR}(t) / (1 + \text{SNR}(t))$ -- this approximates the ELBO weighting and gives more importance to low-SNR (high-noise) timesteps.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# Part 1: Find timestep where SNR = 1
snr_values = compute_snr(sched['alpha_bar'])                      # (1000,)
snr_eq_1_idx = (snr_values - 1.0).abs().argmin().item()           # scalar index
print(f"Timestep closest to SNR=1: t={snr_eq_1_idx + 1}")
print(f"  SNR at that timestep: {snr_values[snr_eq_1_idx]:.4f}")
print(f"  alpha_bar at that timestep: {sched['alpha_bar'][snr_eq_1_idx]:.4f}")
print(f"  (alpha_bar should be ~0.5 when SNR=1, since SNR = ab/(1-ab) = 1 => ab = 0.5)")


# Part 2: SNR-weighted loss
def diffusion_loss_snr_weighted(
    model: nn.Module,
    x_0: torch.Tensor,
    sched: dict,
    T: int = 1000,
) -> torch.Tensor:
    """Compute SNR-weighted diffusion loss.
    
    Weight = SNR(t) / (1 + SNR(t)) = alpha_bar_t  (after simplification!)
    """
    batch_size = x_0.shape[0]
    t = torch.randint(0, T, (batch_size,), device=x_0.device)               # (B,)
    eps = torch.randn_like(x_0)                                               # (B, C, H, W)
    
    sqrt_ab = sched['sqrt_alpha_bar'][t].view(-1, 1, 1, 1)                   # (B, 1, 1, 1)
    sqrt_1mab = sched['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1)       # (B, 1, 1, 1)
    x_t = sqrt_ab * x_0 + sqrt_1mab * eps                                    # (B, C, H, W)
    
    eps_theta = model(x_t, t)                                                 # (B, C, H, W)
    
    # Per-sample MSE
    per_sample_loss = (eps - eps_theta).pow(2).mean(dim=(1, 2, 3))           # (B,)
    
    # Weight by SNR(t) / (1 + SNR(t)) = alpha_bar_t
    weights = sched['alpha_bar'][t]                                           # (B,)
    
    weighted_loss = (weights * per_sample_loss).mean()                        # scalar
    return weighted_loss


torch.manual_seed(42)
loss_simple = diffusion_loss_simple(dummy_model, x_0_batch, sched, T=1000)
loss_snr = diffusion_loss_snr_weighted(dummy_model, x_0_batch, sched, T=1000)
print(f"\nSimple loss:       {loss_simple.item():.4f}")
print(f"SNR-weighted loss: {loss_snr.item():.4f}")
print("SNR-weighted loss is smaller because it downweights high-noise timesteps.")

## 5.9 Variance Schedules

The choice of $\beta_t$ schedule significantly affects training and sample quality. We compare three common schedules.

### Linear Schedule (DDPM)
$$\beta_t = \beta_{\text{start}} + \frac{t-1}{T-1}(\beta_{\text{end}} - \beta_{\text{start}})$$

Simple and effective, but destroys information too quickly at high $t$ -- the transition from "mostly signal" to "mostly noise" is abrupt.

### Cosine Schedule ([Nichol & Dhariwal 2021](https://arxiv.org/abs/2102.09672))

Designed so that $\bar{\alpha}_t$ follows a cosine curve, giving a smoother transition:

$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \qquad f(t) = \cos\!\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2$$

with small offset $s = 0.008$ to prevent $\beta_t$ from being too small near $t=0$.

### Sigmoid Schedule

A smooth S-curve interpolation in log-space:

$$\beta_t = \sigma(-a + 2a \cdot t/T) \cdot (\beta_{\text{end}} - \beta_{\text{start}}) + \beta_{\text{start}}$$

where $\sigma$ is the sigmoid function and $a$ controls the steepness.

In [ ]:
# --- 5.9 Implement and compare variance schedules ---

def linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """Linear schedule: betas linearly interpolated from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, T)  # (T,)


def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    """Cosine schedule from Nichol & Dhariwal 2021.
    
    Defines alpha_bar via cosine, then derives betas.
    """
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * (math.pi / 2)) ** 2      # (T+1,)
    alpha_bar = (f_t[1:] / f_t[0])                                           # (T,)
    # Derive betas: beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}
    alpha_bar_full = torch.cat([torch.tensor([1.0], dtype=torch.float64), alpha_bar])  # (T+1,)
    betas = 1.0 - (alpha_bar_full[1:] / alpha_bar_full[:-1])                 # (T,)
    betas = torch.clamp(betas, min=0.0, max=0.999)                           # clip for stability
    return betas.float()                                                      # (T,)


def sigmoid_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02, steepness: float = 6.0) -> torch.Tensor:
    """Sigmoid schedule: smooth S-curve interpolation."""
    t = torch.linspace(-steepness, steepness, T)                              # (T,)
    betas = torch.sigmoid(t) * (beta_end - beta_start) + beta_start          # (T,)
    return betas                                                              # (T,)


T = 1000
betas_linear = linear_beta_schedule(T)
betas_cosine = cosine_beta_schedule(T)
betas_sigmoid = sigmoid_beta_schedule(T)

sched_linear = compute_schedule(betas_linear)
sched_cosine = compute_schedule(betas_cosine)
sched_sigmoid = compute_schedule(betas_sigmoid)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot betas
for name, s, color in [('Linear', sched_linear, 'tab:blue'), 
                         ('Cosine', sched_cosine, 'tab:orange'),
                         ('Sigmoid', sched_sigmoid, 'tab:green')]:
    axes[0].plot(timesteps.numpy(), s['betas'].numpy(), label=name, color=color)
    axes[1].plot(timesteps.numpy(), s['alpha_bar'].numpy(), label=name, color=color)
    axes[2].semilogy(timesteps.numpy(), compute_snr(s['alpha_bar']).numpy(), label=name, color=color)

axes[0].set_title(r'$\beta_t$')
axes[0].set_xlabel('Timestep')
axes[0].legend()

axes[1].set_title(r'$\bar{\alpha}_t$')
axes[1].set_xlabel('Timestep')
axes[1].legend()

axes[2].set_title('SNR (log scale)')
axes[2].set_xlabel('Timestep')
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- 5.9 Visual comparison: how each schedule noises an image ---

x_0 = make_checkerboard()  # (1, 32, 32)
torch.manual_seed(42)

vis_steps = [0, 200, 400, 600, 800, 1000]
schedules = {'Linear': sched_linear, 'Cosine': sched_cosine, 'Sigmoid': sched_sigmoid}

fig, axes = plt.subplots(3, len(vis_steps), figsize=(16, 8))

for row, (name, s) in enumerate(schedules.items()):
    for col, t in enumerate(vis_steps):
        if t == 0:
            img = x_0[0]
        else:
            torch.manual_seed(42)  # Same noise for fair comparison
            eps = torch.randn_like(x_0)
            t_idx = t - 1
            img = (s['sqrt_alpha_bar'][t_idx] * x_0 + s['sqrt_one_minus_alpha_bar'][t_idx] * eps)[0]
        
        axes[row, col].imshow(img.numpy(), cmap='gray', vmin=-3, vmax=3)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f"t = {t}", fontsize=11)
    axes[row, 0].set_ylabel(name, fontsize=12, rotation=0, labelpad=50)

fig.suptitle("Forward Noising Under Different Schedules (same noise realization)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Exercise 5.9: Implement All Three Schedules and Compare

Implement `linear_beta_schedule`, `cosine_beta_schedule`, and `sigmoid_beta_schedule` from scratch, then plot:
1. $\bar{\alpha}_t$ for all three on one plot
2. The log-SNR $\log(\text{SNR}(t))$ for all three on one plot

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
# (The schedule functions were already implemented above. Here we create the comparison plots.)

T = 1000
schedules_compare = {
    'Linear': compute_schedule(linear_beta_schedule(T)),
    'Cosine': compute_schedule(cosine_beta_schedule(T)),
    'Sigmoid': compute_schedule(sigmoid_beta_schedule(T)),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
t_axis = torch.arange(1, T + 1).numpy()

for name, s in schedules_compare.items():
    ab = s['alpha_bar']
    snr = compute_snr(ab)
    log_snr = torch.log(snr + 1e-10)
    
    axes[0].plot(t_axis, ab.numpy(), label=name)
    axes[1].plot(t_axis, log_snr.numpy(), label=name)

axes[0].set_title(r'$\bar{\alpha}_t$ Comparison')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].legend()

axes[1].set_title(r'$\log\,\mathrm{SNR}(t)$ Comparison')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel(r'$\log\,\mathrm{SNR}$')
axes[1].axhline(y=0.0, color='gray', linestyle='--', alpha=0.5, label='SNR=1')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
print(f"{'Schedule':<10} {'SNR=1 at t':<12} {'alpha_bar[T]':<14} {'alpha_bar[1]':<14}")
print("-" * 50)
for name, s in schedules_compare.items():
    snr = compute_snr(s['alpha_bar'])
    t_snr1 = (snr - 1.0).abs().argmin().item() + 1
    print(f"{name:<10} {t_snr1:<12} {s['alpha_bar'][-1].item():<14.6f} {s['alpha_bar'][0].item():<14.6f}")

---

## Math Reinforcement Exercises

These exercises consolidate the theory from this module through hands-on numerical verification and experimentation.

### Exercise R1: Derive and Verify the $x_t$ Formula

Starting from the single-step formula $x_t = \sqrt{1-\beta_t}\, x_{t-1} + \sqrt{\beta_t}\, \varepsilon_t$, verify numerically that the closed-form $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$ produces the same distribution.

Run the sequential chain many times for a fixed pixel, collect the distribution, and compare to the closed-form distribution (mean and variance).

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

betas_r1 = linear_beta_schedule(T=1000)
sched_r1 = compute_schedule(betas_r1)

x0_val = 0.8  # A fixed pixel value
target_t = 300
n_trials = 10000

# Method A: Sequential chain (many trials)
sequential_samples = []
for _ in range(n_trials):
    x = x0_val
    for step in range(target_t):
        beta = betas_r1[step].item()
        x = math.sqrt(1 - beta) * x + math.sqrt(beta) * torch.randn(1).item()
    sequential_samples.append(x)
sequential_samples = torch.tensor(sequential_samples)  # (n_trials,)

# Method B: Closed-form (many trials)
ab_t = sched_r1['alpha_bar'][target_t - 1].item()
closed_form_samples = math.sqrt(ab_t) * x0_val + math.sqrt(1 - ab_t) * torch.randn(n_trials)  # (n_trials,)

# Compare
print(f"Target: t={target_t}, x_0={x0_val}")
print(f"Theoretical mean:  {math.sqrt(ab_t) * x0_val:.4f}")
print(f"Theoretical var:   {1 - ab_t:.4f}")
print()
print(f"Sequential  -- mean: {sequential_samples.mean():.4f}, var: {sequential_samples.var():.4f}")
print(f"Closed-form -- mean: {closed_form_samples.mean():.4f}, var: {closed_form_samples.var():.4f}")

# Histogram comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sequential_samples.numpy(), bins=60, alpha=0.5, density=True, label='Sequential chain')
ax.hist(closed_form_samples.numpy(), bins=60, alpha=0.5, density=True, label='Closed-form')
ax.set_title(f'Distribution of $x_{{t={target_t}}}$ for $x_0={x0_val}$')
ax.set_xlabel('$x_t$')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise R2: Verify the Posterior $q(x_{t-1} | x_t, x_0)$

The posterior has mean $\tilde{\mu}_t = \frac{\sqrt{\bar{\alpha}_{t-1}} \beta_t}{1 - \bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t$ and variance $\tilde{\beta}_t$.

Verify this by:
1. Fixing $x_0$ and $x_t$
2. Sampling $x_{t-1}$ from the sequential process conditioned on both (by filtering: run the forward chain, keep only runs that land near $x_t$)
3. Comparing to the closed-form Gaussian

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

# We verify the posterior analytically by direct computation, since
# rejection sampling is too expensive for high-dimensional data.
# Instead, we verify that sampling from the posterior and then applying
# one forward step recovers the correct joint distribution.

betas_r2 = linear_beta_schedule(T=1000)
sched_r2 = compute_schedule(betas_r2)

t_idx = 50  # Use a small t for clarity (0-indexed, so this is t=51)
x0_val = torch.tensor([0.7])       # (1,) -- scalar "image"

# Compute x_t using closed-form
torch.manual_seed(99)
eps = torch.randn(1)                # (1,)
x_t_val = sched_r2['sqrt_alpha_bar'][t_idx] * x0_val + sched_r2['sqrt_one_minus_alpha_bar'][t_idx] * eps  # (1,)

# Posterior mean (closed-form)
alpha_bar_t = sched_r2['alpha_bar'][t_idx].item()
alpha_bar_prev = sched_r2['alpha_bar_prev'][t_idx].item()
beta_t = sched_r2['betas'][t_idx].item()
alpha_t = sched_r2['alphas'][t_idx].item()

posterior_mean = (
    (math.sqrt(alpha_bar_prev) * beta_t) / (1 - alpha_bar_t) * x0_val
    + (math.sqrt(alpha_t) * (1 - alpha_bar_prev)) / (1 - alpha_bar_t) * x_t_val
)
posterior_var = sched_r2['posterior_variance'][t_idx].item()

print(f"At t_idx={t_idx}:")
print(f"  x_0 = {x0_val.item():.4f}")
print(f"  x_t = {x_t_val.item():.4f}")
print(f"  Posterior mean = {posterior_mean.item():.4f}")
print(f"  Posterior var  = {posterior_var:.6f}")
print(f"  Posterior std  = {math.sqrt(posterior_var):.6f}")

# Sample many x_{t-1} from the posterior and verify statistics
n_samples = 50000
posterior_samples = posterior_mean + math.sqrt(posterior_var) * torch.randn(n_samples)  # (n_samples,)

# Now verify: if we take these x_{t-1} and apply one forward step,
# we should get a distribution centered near x_t
x_t_reconstructed = math.sqrt(1 - beta_t) * posterior_samples + math.sqrt(beta_t) * torch.randn(n_samples)

print(f"\nVerification: forward step from posterior samples should match x_t")
print(f"  x_t (original):     {x_t_val.item():.4f}")
print(f"  E[forward(x_{{t-1}})]:{x_t_reconstructed.mean().item():.4f}")

# Also verify posterior sample statistics match
print(f"\nPosterior sample statistics:")
print(f"  Empirical mean: {posterior_samples.mean().item():.4f}  (expected {posterior_mean.item():.4f})")
print(f"  Empirical var:  {posterior_samples.var().item():.6f}  (expected {posterior_var:.6f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(posterior_samples.numpy(), bins=80, density=True, alpha=0.6, label='Posterior samples')
# Overlay the theoretical Gaussian
x_range = torch.linspace(posterior_mean.item() - 4*math.sqrt(posterior_var),
                          posterior_mean.item() + 4*math.sqrt(posterior_var), 200)
pdf = torch.exp(-0.5 * ((x_range - posterior_mean.item()) / math.sqrt(posterior_var))**2) / (math.sqrt(2*math.pi*posterior_var))
ax.plot(x_range.numpy(), pdf.numpy(), 'r-', lw=2, label='Theoretical Gaussian')
ax.set_title(f'Posterior $q(x_{{t-1}} | x_t, x_0)$ at t={t_idx+1}')
ax.set_xlabel('$x_{t-1}$')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise R3: Find the Timestep Where SNR = 1

For each of the three schedules (linear, cosine, sigmoid), find the exact timestep where the signal-to-noise ratio equals 1 (i.e., $\bar{\alpha}_t = 0.5$). Verify by computing $\bar{\alpha}_t$ at that timestep.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

T = 1000
schedule_fns = {
    'Linear': linear_beta_schedule,
    'Cosine': cosine_beta_schedule,
    'Sigmoid': sigmoid_beta_schedule,
}

print(f"{'Schedule':<10} {'t (SNR=1)':<12} {'alpha_bar':<12} {'SNR':<12}")
print("-" * 46)

for name, fn in schedule_fns.items():
    betas = fn(T)
    s = compute_schedule(betas)
    snr = compute_snr(s['alpha_bar'])
    
    # Find index closest to SNR = 1
    idx = (snr - 1.0).abs().argmin().item()
    
    print(f"{name:<10} {idx + 1:<12} {s['alpha_bar'][idx].item():<12.6f} {snr[idx].item():<12.6f}")

print("\nNote: SNR=1 means alpha_bar=0.5 exactly. The linear schedule reaches")
print("this crossover much earlier than cosine, meaning it wastes timesteps")
print("in the pure-noise regime where the model gets little learning signal.")

### Exercise R4: Forward Noising Visualization on Synthetic Images

Create a grid of synthetic test images (gradient, circles, stripes) and visualize the forward noising process at multiple timesteps using the cosine schedule.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

def make_gradient(size: int = 32) -> torch.Tensor:
    """Horizontal gradient from -1 to 1, shape (1, size, size)."""
    grad = torch.linspace(-1, 1, size).unsqueeze(0).repeat(size, 1)  # (size, size)
    return grad.unsqueeze(0)  # (1, size, size)

def make_circle(size: int = 32, radius: float = 0.6) -> torch.Tensor:
    """Centered circle, shape (1, size, size). Inside=1, outside=-1."""
    coords = torch.linspace(-1, 1, size)
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')          # (size, size) each
    dist = torch.sqrt(xx**2 + yy**2)                                 # (size, size)
    img = torch.where(dist < radius, torch.ones_like(dist), -torch.ones_like(dist))
    return img.unsqueeze(0)  # (1, size, size)

def make_stripes(size: int = 32, freq: int = 4) -> torch.Tensor:
    """Vertical stripes, shape (1, size, size)."""
    x = torch.linspace(0, 2 * math.pi * freq, size)
    stripes = torch.sin(x).unsqueeze(0).repeat(size, 1)             # (size, size)
    return stripes.unsqueeze(0)  # (1, size, size)


images = {
    'Checkerboard': make_checkerboard(),
    'Gradient': make_gradient(),
    'Circle': make_circle(),
    'Stripes': make_stripes(),
}

sched_cos = compute_schedule(cosine_beta_schedule(1000))
vis_steps = [0, 100, 300, 500, 700, 900]

fig, axes = plt.subplots(len(images), len(vis_steps), figsize=(16, 10))
torch.manual_seed(42)

for row, (name, img) in enumerate(images.items()):
    for col, t in enumerate(vis_steps):
        if t == 0:
            noisy = img[0]
        else:
            torch.manual_seed(42 + row)  # Different noise per image, but reproducible
            eps = torch.randn_like(img)
            t_idx = t - 1
            noisy = (sched_cos['sqrt_alpha_bar'][t_idx] * img
                     + sched_cos['sqrt_one_minus_alpha_bar'][t_idx] * eps)[0]
        
        axes[row, col].imshow(noisy.numpy(), cmap='gray', vmin=-2, vmax=2)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f"t = {t}", fontsize=11)
    axes[row, 0].set_ylabel(name, fontsize=11, rotation=0, labelpad=70)

fig.suptitle("Forward Noising (Cosine Schedule) on Synthetic Images", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Exercise R5: Schedule Comparison with a Tiny MLP on 2D Data

Train a tiny MLP to denoise 2D points from a simple distribution (circle) using the simplified loss. Compare training with linear vs. cosine schedules by plotting loss curves.

In [ ]:
# ✅ SOLUTION — try the exercise above before running this

class TinyDiffusionMLP(nn.Module):
    """A small MLP that predicts noise for 2D data.
    
    Input: (x_t [2D], t_embedding [16D]) -> predicted noise [2D]
    """
    def __init__(self, T: int = 1000, embed_dim: int = 16, hidden: int = 128):
        super().__init__()
        # Simple sinusoidal timestep embedding
        self.T = T
        self.embed_dim = embed_dim
        self.net = nn.Sequential(
            nn.Linear(2 + embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 2),
        )
    
    def timestep_embedding(self, t: torch.Tensor) -> torch.Tensor:
        """Simple sinusoidal embedding of timestep t."""
        half = self.embed_dim // 2
        freqs = torch.exp(-math.log(self.T) * torch.arange(half, device=t.device).float() / half)  # (half,)
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)      # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, embed_dim)
    
    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.timestep_embedding(t)                        # (B, embed_dim)
        inp = torch.cat([x_t, t_emb], dim=-1)                    # (B, 2 + embed_dim)
        return self.net(inp)                                       # (B, 2)


def sample_circle_2d(n: int) -> torch.Tensor:
    """Sample n points uniformly on a unit circle."""
    angles = torch.rand(n) * 2 * math.pi                          # (n,)
    x = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)  # (n, 2)
    return x


def train_2d_diffusion(schedule_name: str, betas: torch.Tensor, n_steps: int = 2000, batch_size: int = 256):
    """Train a tiny 2D diffusion model and return the loss history."""
    T = len(betas)
    s = compute_schedule(betas)
    
    # Move schedule to device
    sqrt_ab = s['sqrt_alpha_bar'].to(device)                       # (T,)
    sqrt_1mab = s['sqrt_one_minus_alpha_bar'].to(device)           # (T,)
    
    model = TinyDiffusionMLP(T=T).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    
    losses = []
    for step in range(n_steps):
        x_0 = sample_circle_2d(batch_size).to(device)              # (B, 2)
        t = torch.randint(0, T, (batch_size,), device=device)     # (B,)
        eps = torch.randn_like(x_0)                                # (B, 2)
        
        x_t = sqrt_ab[t].unsqueeze(-1) * x_0 + sqrt_1mab[t].unsqueeze(-1) * eps  # (B, 2)
        
        eps_theta = model(x_t, t)                                  # (B, 2)
        loss = F.mse_loss(eps_theta, eps)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
    
    return losses


# Train with both schedules
torch.manual_seed(42)
T = 200  # Smaller T for fast training on 2D data

losses_linear = train_2d_diffusion('Linear', linear_beta_schedule(T), n_steps=2000)
torch.manual_seed(42)
losses_cosine = train_2d_diffusion('Cosine', cosine_beta_schedule(T), n_steps=2000)

# Plot loss curves
fig, ax = plt.subplots(figsize=(10, 5))

# Smooth with moving average
window = 50
def smooth(losses, w=window):
    return np.convolve(losses, np.ones(w)/w, mode='valid')

ax.plot(smooth(losses_linear), label='Linear schedule', color='tab:blue', alpha=0.8)
ax.plot(smooth(losses_cosine), label='Cosine schedule', color='tab:orange', alpha=0.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('MSE Loss (smoothed)')
ax.set_title('2D Diffusion Training: Linear vs Cosine Schedule')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final loss (linear, avg last 100): {np.mean(losses_linear[-100:]):.4f}")
print(f"Final loss (cosine, avg last 100): {np.mean(losses_cosine[-100:]):.4f}")